# 5. End-to-end RAG

**Learning objective:** follow one family intake through the entire online path,
and understand the grounding check that decides whether a generated pathway is
allowed to reach a family at all.

**Where this fits:** this notebook joins everything from notebooks 1 through 4.

```
FamilyIntake -> retrieval query -> retrieved records -> context -> generation
             -> citation check -> GroundedPathwayResult
```

The whole flow is exercised offline here, with small fakes standing in for the
retriever and the generator. The service being tested is the real one.

In [ ]:
from mosaic_pathway.models import (
    ChildProfile,
    CommunitySuggestion,
    FamilyIntake,
    LearningPathway,
    ResourceRecommendation,
    RetrievalResult,
    RetrievedRecord,
    RhythmPractice,
    SourceRecord,
)
from mosaic_pathway.query_builder import build_retrieval_query
from mosaic_pathway.rag import (
    DEFAULT_MAX_PER_SOURCE,
    DEFAULT_TOP_K,
    GroundingError,
    MosaicPathwayService,
    build_context,
    cited_source_ids,
)

print(
    "service defaults: top_k =",
    DEFAULT_TOP_K,
    "max_per_source =",
    DEFAULT_MAX_PER_SOURCE,
)

## Step 1: a deterministic query from a structured intake

The retriever needs a string; the family gave us a structured form. Turning one
into the other is done by a plain function with no model call in it, so the same
intake always produces the same query, and a retrieval result is reproducible.

The query is written in the family's own terms rather than as keywords, because
the embedding model was trained on natural language, not on search syntax.

In [ ]:
intake = FamilyIntake(
    children=[
        ChildProfile(
            label="older child",
            age=12,
            interests=["animals", "drawing"],
            learning_needs=["movement breaks"],
        )
    ],
    leaving_behind=["rigid daily schedules"],
    wants_to_preserve=["reading together after dinner"],
    wants_to_add=["more time outdoors"],
    family_values=["curiosity", "gentleness"],
    practical_constraints=["one working parent at home"],
    additional_context="We are in our first year of self-directed learning.",
)

query = build_retrieval_query(intake)

print(query)
print()
print("deterministic:", query == build_retrieval_query(intake))

## Step 2: fakes for the two expensive dependencies

`MosaicPathwayService` takes a retriever and a generator in its constructor and
calls two methods on them. Nothing else. That constructor is the seam that makes
the service testable without Qdrant or Azure.

The fakes below are notebook scaffolding, not an alternative implementation: they
return the real domain models, and the orchestration under test is production
code.

In [ ]:
def make_record(index: int, text: str) -> SourceRecord:
    return SourceRecord(
        source_id=f"synthetic-guide-{index:04d}",
        title=f"Synthetic guide chunk {index}",
        source_file="synthetic-guide.docx",
        content_type="practical_guidance",
        authority_type="mosaic_guidance",
        topics=["rhythm", "outdoor learning"],
        text=text,
    )


CORPUS = [
    make_record(
        7,
        "Families often start with one small repeated practice rather than a full timetable.",
    ),
    make_record(
        11, "An interest catalog is a shared list of what a child keeps returning to."
    ),
    make_record(
        19,
        "A short walk before the day begins often changes the tone of everything after it.",
    ),
]


class FakeRetriever:
    """Returns a fixed set of records and records the query it was given."""

    def __init__(self, records: list[SourceRecord]) -> None:
        self.records = records
        self.queries: list[str] = []

    def retrieve(self, query: str, top_k: int, max_per_source: int) -> RetrievalResult:
        self.queries.append(query)
        scored = [
            RetrievedRecord(record=record, score=0.9 - 0.1 * position)
            for position, record in enumerate(self.records)
        ]

        return RetrievalResult(query=query, records=scored[:top_k])


class FakeGenerator:
    """Returns a pathway citing whichever source ids it is configured with."""

    def __init__(self, cited_ids: list[str]) -> None:
        self.cited_ids = cited_ids
        self.contexts: list[list[dict[str, str]]] = []

    def generate(
        self, intake: FamilyIntake, context: list[dict[str, str]]
    ) -> LearningPathway:
        self.contexts.append(context)
        child = intake.children[0]

        return LearningPathway(
            family_reflection=(
                f"Your {child.label} loves {child.interests[0]}, and you are moving "
                "away from a rigid timetable toward more time outdoors."
            ),
            starting_rhythm=[
                RhythmPractice(
                    timing="Most mornings",
                    practice="Take a short walk before the day begins.",
                    why_it_fits="It adds outdoor time without adding a schedule.",
                ),
                RhythmPractice(
                    timing="Once a week",
                    practice="Add one page to a shared drawing journal.",
                    why_it_fits="It connects the walk to drawing.",
                ),
            ],
            resources=[
                ResourceRecommendation(
                    title="Start an interest catalog",
                    why_it_fits="It gives the interest in animals somewhere to grow.",
                    source_id=self.cited_ids[0],
                    url=None,
                ),
                ResourceRecommendation(
                    title="Build a gentle weekly rhythm",
                    why_it_fits="It replaces the timetable you are leaving behind.",
                    source_id=self.cited_ids[1],
                    url=None,
                ),
            ],
            community_suggestion=CommunitySuggestion(
                suggestion="Visit one informal nature meetup this month.",
                why_it_fits="It is low pressure and matches your outdoor goal.",
                source_id=self.cited_ids[2],
            ),
            closing_note="Go slowly. One walk and one journal page is a real start.",
        )


print("fakes defined for", len(CORPUS), "synthetic records")

## Step 3: the success path

The service builds the query, retrieves, converts records into context,
generates, checks the citations, and returns everything together.

In [ ]:
retriever = FakeRetriever(CORPUS)
generator = FakeGenerator(
    ["synthetic-guide-0011", "synthetic-guide-0019", "synthetic-guide-0007"]
)
service = MosaicPathwayService(retriever, generator)

result = service.generate_pathway(intake)

print("service reused the deterministic query:", retriever.queries[0] == query)
print("retrieved order:", [item.record.source_id for item in result.retrieved_records])
print("scores         :", [round(item.score, 2) for item in result.retrieved_records])
print("cited ids      :", cited_source_ids(result.pathway))

Retrieval order is preserved through the whole flow. The context list given to
the generator is in ranked order, and `retrieved_records` on the result is the
same list, so the evidence shown to a family in notebook 7 matches the ranking
the retriever produced.

In [ ]:
context = build_context(result.retrieved_records)

print("context entries:", len(context))
print("keys per entry :", sorted(context[0]))
print()

for entry in context:
    print(f"{entry['source_id']} ({entry['inventory_source_id']}) | {entry['title']}")
    print(f"   {entry['text'][:80]}...")

Each context entry carries both identities from notebook 1. The chunk id is what
the model must cite; the inventory id is there so the model can tell when two
passages come from the same document.

The context is also the exact private text sent to Azure OpenAI. That is the one
place in the system where Mosaic material leaves the machine, which is why the
boundary is a small explicit function rather than an ad hoc dictionary built
inside the generator.

## Step 4: the grounding check

After generation, the service collects every `source_id` the pathway cites and
subtracts the set of ids that were actually retrieved. If anything is left, the
pathway is discarded and `GroundingError` is raised.

This catches a specific and common failure: the model inventing a plausible id,
or reusing one it saw in an earlier turn. It is a cheap, deterministic check, and
it runs on every request.

In [ ]:
hallucinating_generator = FakeGenerator(
    ["synthetic-guide-0011", "synthetic-guide-0042", "synthetic-guide-0007"]
)
strict_service = MosaicPathwayService(retriever, hallucinating_generator)

try:
    strict_service.generate_pathway(intake)
except GroundingError as error:
    print("GroundingError:", error)

## Step 5: the other guard, and what neither guard covers

If retrieval comes back empty, the service refuses to generate at all. A pathway
built from no evidence would be pure invention with a confident tone.

In [ ]:
empty_service = MosaicPathwayService(FakeRetriever([]), generator)

try:
    empty_service.generate_pathway(intake)
except RuntimeError as error:
    print("RuntimeError:", error)

Both checks are about **provenance**, not **support**. A pathway can pass every
check here and still be wrong, because a valid id only proves the passage was
retrieved, not that it says what the recommendation claims.

Closing that gap needs either a semantic entailment check or a person. This
project chose a person, and notebook 6 describes the rubric they use.

## Locating a failure

When a result looks wrong, the `GroundedPathwayResult` contains enough to assign
the failure to one owner rather than guessing.

| Symptom | Look at | Likely owner |
| --- | --- | --- |
| The query does not describe the family | `retrieval_query` | intake or query builder |
| The query is fine, evidence is off-topic | `retrieved_records` | retrieval or corpus gap |
| Evidence is good, pathway ignores it | `pathway` versus context | generation |
| Pathway cites unretrieved ids | `GroundingError` message | generation |
| Everything is on-topic but generic | interests in `pathway` prose | prompt or generation |

The first question is always whether the evidence was good. Debugging a
generation problem that is actually a retrieval problem wastes a lot of time.

## Optional live Azure OpenAI section

The cells below are the only ones in this notebook that touch Azure or the local
index. They need a built vector store, the local embedding model, and a signed-in
Azure session, and they are disabled by default.

Close any running Streamlit app or API process first: local Qdrant allows a
single process to hold the directory.

The output prints the retrieval query, the retrieved ids and titles, and the
family-facing pathway. It never prints a full source passage, and nothing is
written to a tracked file.

In [ ]:
RUN_LIVE_AZURE = False

print("live Azure section enabled:", RUN_LIVE_AZURE)

In [ ]:
if RUN_LIVE_AZURE:
    from mosaic_pathway.embeddings import LocalEmbeddingModel
    from mosaic_pathway.generation import AzureOpenAIPathwayGenerator
    from mosaic_pathway.retrieval import MosaicRetriever
    from mosaic_pathway.settings import load_settings
    from mosaic_pathway.vector_store import MosaicVectorStore

    with MosaicVectorStore() as store:
        live_service = MosaicPathwayService(
            MosaicRetriever(LocalEmbeddingModel(), store),
            AzureOpenAIPathwayGenerator(load_settings()),
        )
        live_result = live_service.generate_pathway(intake)

    print("retrieval query:")
    print(live_result.retrieval_query)
    print()
    print("evidence:")

    for item in live_result.retrieved_records:
        print(f"  {item.score:.3f} {item.record.source_id} | {item.record.title}")

    print()
    print(live_result.pathway.family_reflection)
    print()

    for practice in live_result.pathway.starting_rhythm:
        print(f"{practice.timing}: {practice.practice}")

    print()

    for resource in live_result.pathway.resources:
        print(f"{resource.title} [{resource.source_id}]")

    print()
    print(live_result.pathway.closing_note)
else:
    print(
        "Skipped: set RUN_LIVE_AZURE to True to run against Azure and the local index."
    )

## Key takeaways

* The intake to query step is a deterministic function, which keeps retrieval
  reproducible and debuggable.
* `MosaicPathwayService` depends on two method shapes, so the whole orchestration
  runs offline with small fakes.
* `build_context` is the single place private text crosses the network boundary.
* Two deterministic guards run on every request: no evidence means no generation,
  and an unretrieved citation means the pathway is discarded.
* Provenance is not support; valid citations do not make a claim true.
* The grounded result carries enough context to assign a failure to retrieval or
  to generation instead of guessing.

## Next

Notebook 6 turns these one-off observations into a repeatable evaluation suite
over a fixed set of family cases.